**Подготовка DataSet результатов продаж**

**Цель:** сформировать окна размеростью 12 месяцев, для каждого окна определить целевую переменную.

**Целевая переменная** -- величина продаж в прогнозируемом месяце.

In [28]:
import pandas as pd
import datetime
from dateutil.relativedelta import relativedelta
from typing import List

In [29]:
df_xls = pd.read_excel('temp_8_9_10_итог_с_июлем.xlsx')
df_xls.head()

,location_mdlp_id,gc_address,gc_lat,gc_lng,territory,geoball,2024-01-01,2024-02-01,2024-03-01,2024-04-01,...,2025-03-01,2025-04-01,2025-05-01,2025-06-01,2025-07-01,class,method,2025-08-01,2025-09-01,2025-10-01
0,166.0,"Россия, Москва, Митинская улица, 43",55.849565,37.352767,KAM Moscow_3,9.0,18.0,7.0,14.0,15.0,...,16.0,30,19.0,27.0,20.0,Стабильная,LR,19,19.0,20.0
1,169.0,"Россия, Москва, улица Маршала Федоренко, 12",55.881722,37.488844,KAM Moscow_15,3.0,13.0,18.0,16.0,16.0,...,34.0,38,21.0,27.0,18.0,Стабильная,LR,22,22.0,21.0
2,170.0,"Россия, Москва, Лукинская улица, 5к1",55.645517,37.342373,KAM SpCardio Central_3,5.0,15.0,3.0,13.0,6.0,...,13.0,26,10.0,14.0,18.0,Стабильная,LR,21,22.0,23.0
3,171.0,"Россия, Москва, Сходненская улица, 36/11",55.842994,37.439625,KAM Moscow_3,8.0,NaN,NaN,NaN,NaN,...,0.0,0,3.0,2.0,0.0,Нестабильная,Median+SMA,1,1.0,1.0
4,172.0,"Россия, Москва, Уссурийская улица, 9",55.823980,37.819082,KAM Moscow_VAO_2,7.0,11.0,6.0,14.0,9.0,...,29.0,23,36.0,39.0,26.0,Стабильная,LR,30,31.0,33.0


In [30]:
date_0 = datetime.datetime.strptime('2024-01-01', '%Y-%m-%d')
ls_date = [ (date_0 + relativedelta(months=num_mes)).strftime('%Y-%m-%d') for num_mes in range(19)]

df_stable = df_xls[df_xls['class']=='Стабильная'][ls_date]
filtered_df = df_stable.dropna(subset=ls_date)
ls_rec_dict = filtered_df.to_dict('records')

In [31]:
ls_rec_tuples = [list(rec.items()) for rec in ls_rec_dict]
print(f'{len(ls_rec_tuples[0])}, last_val: {ls_rec_tuples[0][-1]}')

19, last_val: ('2025-07-01', 20.0)


In [32]:
def sliding_windows(data: List, window: int, step: int = 1) -> List[List]:
    """

    :param data: исходный список
    :param window: размер окна
    :param step: шаг сдвига окна
    :return: Список окон.
    """
    # значение последнего месяца
    last_val = data.pop()
    
    ls_window = []
    n = len(data)
    for i in range(0, n, step):
        w = data[i:i + window]
        if len(w) < window:
            break
        ls_window.append(w)

    # Заполнение "прогнозированное значение"
    result = []
    while len(ls_window):
        one_window = ls_window.pop()
        dc = {'window': one_window, 'last_val': last_val}
        last_val = one_window[-1]
        result.append(dc)

    return result

# пример
all_windows = []
for prod_aptek in ls_rec_tuples:
    ls_windows = sliding_windows(prod_aptek, window=12, step=1)
    all_windows.extend(ls_windows)


In [35]:
print(f'{len(all_windows)=}')
all_windows[0]


len(all_windows)=4452


{'window': [('2024-07-01', 23.0),
  ('2024-08-01', 25.0),
  ('2024-09-01', 19.0),
  ('2024-10-01', 27.0),
  ('2024-11-01', 12.0),
  ('2024-12-01', 20.0),
  ('2025-01-01', 21.0),
  ('2025-02-01', 20.0),
  ('2025-03-01', 16.0),
  ('2025-04-01', 30),
  ('2025-05-01', 19.0),
  ('2025-06-01', 27.0)],
 'last_val': ('2025-07-01', 20.0)}